# EEG: strictly out-of-fold embeddings + controlled leakage comparison

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not on Colab or already mounted:", e)

In [ ]:
PROJECT    = "/content/drive/MyDrive/Depression_FL_Project"
EEG_DIR    = f"{PROJECT}/data/processed/eeg"            # X_/y_/ids_{train,val,test}.npy (windowed); X_train is MISSING on your Drive
WINDOWS_CACHE = f"{PROJECT}/revision_r1/eeg_windows_raw.npz"  # all 2130 un-normalised windows (rebuilt from raw MODMA on Oct 10)
RAW_MODMA  = f"{PROJECT}/data/raw/MODMA"                 # raw *_still.txt recordings (used only if the cache is missing)
META_PATH  = f"{PROJECT}/data/raw/MODMA/subjects_information_EEG_3channels_resting_lanzhou_2015.xlsx"
R1_OUT     = f"{PROJECT}/fl/revision_r1"
PARTITIONS = f"{R1_OUT}/partitions.csv"
EMB_OUT    = f"{R1_OUT}/eeg_oof"

ENC_SEEDS  = [42, 7, 123, 2024, 99]  # encoder seeds averaged per partition (same seeds as
EPOCHS, PATIENCE, LR, WD, REG_WEIGHT, BATCH = 60, 15, 1e-3, 1e-4, 0.3, 64   # as in the original notebook
PHQ9_MAX   = 27.0                     # fixed scale maximum (no data-dependent normalisation)

RUN_SECTION_A = True     # out-of-fold embeddings for every partition
RUN_SECTION_B = True     # controlled leakage comparison
LEAK_SEEDS    = [42, 7, 123, 2024, 99]
LEAK_FOLDS    = 5
INNER_VAL_FRAC = 0.15    # inner early-stopping split inside each training fold
os.makedirs(EMB_OUT, exist_ok=True)
for p in (EEG_DIR, WINDOWS_CACHE, RAW_MODMA, META_PATH, PARTITIONS):
    print(f"  exists={os.path.exists(p)}  {p}")

In [ ]:
import random, json, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE)

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# The windows are therefore taken, in this order of preference, from:
import glob, subprocess, sys

def norm_id(x):
    """Canonical MODMA subject id: '02010001', '2010001', '2010001.0' -> '2010001'."""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x.lstrip("0") or "0"

meta = pd.read_excel(META_PATH, sheet_name="Sheet1").iloc[:, :11]
meta.columns = ["subject_id", "type", "age", "gender", "education_years",
                "PHQ9", "CTQ_SF", "LES", "SSRS", "GAD7", "PSQI"]
meta = meta.dropna(subset=["subject_id"])
meta["sid"] = meta["subject_id"].map(norm_id)
phq9 = dict(zip(meta.sid, meta.PHQ9.astype(float)))

def rebuild_from_raw_txt():
    try:
        import pywt
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
        import pywt
    from scipy.signal import butter, filtfilt
    FS, EP, STEP, MAXS = 250, 1000, 500, 80 * 250
    b, a = butter(4, [1.0 / (FS / 2), 45.0 / (FS / 2)], btype="band")
    files = {norm_id(os.path.basename(f).split("_")[0]): f
             for f in glob.glob(f"{RAW_MODMA}/**/*.txt", recursive=True)}
    def load_txt(fp):
        try:
            raw = pd.read_csv(fp, header=None, sep=r"\s+", dtype=str)
        except Exception:
            raw = pd.read_csv(fp, header=None, dtype=str)
        def col(v):
            try:
                return v.apply(lambda x: int(str(x).strip(), 16)).astype(np.float32)
            except ValueError:
                return pd.to_numeric(v, errors="coerce").astype(np.float32)
        arrs = [col(raw.iloc[:, i].dropna()).values for i in range(3)]
        n = min(len(x) for x in arrs)
        return np.stack([x[:n] for x in arrs])[:, :MAXS]
    def denoise(eeg):
        out = np.zeros_like(eeg)
        for ch in range(eeg.shape[0]):
            cf = pywt.wavedec(eeg[ch], "db4", level=4)
            sg = np.median(np.abs(cf[-1])) / 0.6745
            th = 3.0 * sg * np.sqrt(2 * np.log(len(eeg[ch])))
            out[ch] = pywt.waverec([cf[0]] + [pywt.threshold(c, th, mode="soft") for c in cf[1:]], "db4")[:eeg.shape[1]]
        return out.astype(np.float32)
    Xs, ys, ids = [], [], []
    for _, row in meta.iterrows():
        fp = files.get(row.sid)
        if fp is None:
            print("  no recording for", row.sid); continue
        eeg = load_txt(fp)
        eeg = (eeg - eeg.mean(1, keepdims=True)).astype(np.float32)
        eeg = denoise(filtfilt(b, a, eeg, axis=1).astype(np.float32))
        ep = np.stack([eeg[:, s:s + EP] for s in range(0, eeg.shape[1] - EP + 1, STEP)]).astype(np.float32)
        pk = np.max(np.abs(ep), axis=(1, 2))
        ep = ep[pk < pk.mean() + 5.0 * pk.std()]
        Xs.append(ep); ys += [1 if str(row.type).strip().upper() == "MDD" else 0] * len(ep); ids += [row.sid] * len(ep)
    return np.concatenate(Xs), np.array(ys), np.array(ids)

have = set(os.listdir(EEG_DIR)) if os.path.isdir(EEG_DIR) else set()
SPL = ("train", "val", "test")
if all(f"X_{sp}_raw.npy" in have and f"ids_{sp}_raw.npy" in have for sp in SPL):
    X_all = np.concatenate([np.load(f"{EEG_DIR}/X_{sp}_raw.npy") for sp in SPL])
    y_all = np.concatenate([np.load(f"{EEG_DIR}/y_{sp}_raw.npy") for sp in SPL])
    ids_all = np.concatenate([np.load(f"{EEG_DIR}/ids_{sp}_raw.npy", allow_pickle=True) for sp in SPL])
    source = "X_*_raw.npy"
elif all(f"X_{sp}.npy" in have for sp in SPL) and {"norm_mean.npy", "norm_std.npy"} <= have:
    mu0 = np.load(f"{EEG_DIR}/norm_mean.npy").astype(np.float32).reshape(1, -1, 1)
    sd0 = np.load(f"{EEG_DIR}/norm_std.npy").astype(np.float32).reshape(1, -1, 1)
    X_all = np.concatenate([np.load(f"{EEG_DIR}/X_{sp}.npy").astype(np.float32) * sd0 + mu0 for sp in SPL])
    y_all = np.concatenate([np.load(f"{EEG_DIR}/y_{sp}.npy") for sp in SPL])
    ids_all = np.concatenate([np.load(f"{EEG_DIR}/ids_{sp}.npy", allow_pickle=True) for sp in SPL])
    source = "X_*.npy (un-normalised)"
elif os.path.exists(WINDOWS_CACHE):
    z = np.load(WINDOWS_CACHE, allow_pickle=True)
    X_all, y_all, ids_all = z["X"], z["y"], z["ids"]
    source = WINDOWS_CACHE
else:
    print("No window files found -> rebuilding from raw MODMA recordings (~3-5 min)")
    X_all, y_all, ids_all = rebuild_from_raw_txt()
    os.makedirs(os.path.dirname(WINDOWS_CACHE), exist_ok=True)
    np.savez(WINDOWS_CACHE, X=X_all, y=y_all, ids=ids_all)
    source = f"rebuilt from raw txt, cached to {WINDOWS_CACHE}"

X_all = X_all.astype(np.float32)
y_all = y_all.astype(np.int64)                       # MDD (1) vs HC (0), per window
ids_all = np.array([norm_id(i) for i in ids_all])
unique_sids = np.unique(ids_all)
y_diag = {s: int(y_all[ids_all == s][0]) for s in unique_sids}
missing = [s for s in unique_sids if s not in phq9]
assert not missing, f"PHQ-9 missing for {missing}"
print(f"source: {source}")
print(f"windows={len(X_all)} shape={X_all.shape} | subjects={len(unique_sids)} "
      f"| MDD={sum(y_diag.values())} HC={len(unique_sids)-sum(y_diag.values())}")
assert X_all.shape[1:] == (3, 1000), X_all.shape

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, n_channels=3, reduction=1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(n_channels, max(1, n_channels // reduction)), nn.ReLU(),
                                nn.Linear(max(1, n_channels // reduction), n_channels), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=2)).unsqueeze(2)

class MultiScaleCNN(nn.Module):
    def __init__(self, in_ch=3, out_ch=32):
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k // 2),
                                                     nn.BatchNorm1d(out_ch), nn.GELU()) for k in [3, 7, 15]])
    def forward(self, x):
        outs = [b(x) for b in self.branches]
        L = min(o.shape[2] for o in outs)
        return torch.cat([o[:, :, :L] for o in outs], dim=1)

class AttentionPool(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.scorer = nn.Sequential(nn.Linear(d_model, 32), nn.Tanh(), nn.Linear(32, 1, bias=False))
    def forward(self, x):
        w = torch.softmax(self.scorer(x), dim=1)
        return (x * w).sum(dim=1)

class MSCA_CNN_GRU(nn.Module):
    def __init__(self, n_channels=3, cnn_out=32, gru_hidden=64, embed_dim=128, dropout=0.4):
        super().__init__()
        self.ch_attn = ChannelAttention(n_channels)
        self.ms_cnn = MultiScaleCNN(n_channels, cnn_out)
        self.pool1d = nn.MaxPool1d(2)
        self.cnn_norm = nn.LayerNorm(cnn_out * 3)
        self.gru = nn.GRU(input_size=cnn_out * 3, hidden_size=gru_hidden, num_layers=1,
                          batch_first=True, bidirectional=True)
        self.attn_pool = AttentionPool(gru_hidden * 2)
        self.proj = nn.Sequential(nn.Linear(gru_hidden * 2, embed_dim), nn.LayerNorm(embed_dim),
                                  nn.GELU(), nn.Dropout(dropout))
        self.classifier = nn.Linear(embed_dim, 1)
        self.regressor = nn.Linear(embed_dim, 1)
    def embed(self, x):
        x = self.pool1d(self.ms_cnn(self.ch_attn(x))).transpose(1, 2)
        x, _ = self.gru(self.cnn_norm(x))
        return self.proj(self.attn_pool(x))
    def forward(self, x):
        e = self.embed(x)
        return self.classifier(e).squeeze(1), torch.sigmoid(self.regressor(e)).squeeze(1)

In [ ]:
# ---------- training helpers ----------
def _norm(model, X):
    """z-score with the per-channel mean/SD of the encoder's own training windows."""
    return torch.from_numpy((X - model.mu) / model.sd).float().to(DEVICE)

@torch.no_grad()
def window_probs(model, X):
    model.eval(); out = []
    for s in range(0, len(X), 512):
        out.append(torch.sigmoid(model(_norm(model, X[s:s + 512]))[0]).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def window_embeddings(model, X):
    model.eval(); out = []
    for s in range(0, len(X), 512):
        out.append(model.embed(_norm(model, X[s:s + 512])).cpu().numpy())
    return np.concatenate(out)

def subject_scores(probs, ids):
    sids = np.unique(ids)
    p = np.array([probs[ids == s].mean() for s in sids])
    return sids, p

def train_encoder(tr_idx, es_idx, seed, es_level="subject"):
    """Window-level training on windows tr_idx; early stopping on windows es_idx."""
    set_all_seeds(seed)
    mu = X_all[tr_idx].mean(axis=(0, 2), keepdims=True)            # (1, 3, 1), training windows only
    sd = X_all[tr_idx].std(axis=(0, 2), keepdims=True) + 1e-8
    Xtr = torch.from_numpy((X_all[tr_idx] - mu) / sd).float().to(DEVICE)
    ytr = torch.from_numpy(y_all[tr_idx].astype(np.float32)).to(DEVICE)
    ptr = torch.from_numpy(np.array([phq9[s] / PHQ9_MAX for s in ids_all[tr_idx]], dtype=np.float32)).to(DEVICE)
    n_pos = int(y_all[tr_idx].sum()); n_neg = len(tr_idx) - n_pos
    model = MSCA_CNN_GRU().to(DEVICE)
    model.mu, model.sd = mu, sd
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    crit_cls = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE))
    crit_reg = nn.MSELoss()
    best, best_state, pat = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(tr_idx), device=DEVICE)
        for s in range(0, len(tr_idx), BATCH):
            b = perm[s:s + BATCH]
            opt.zero_grad()
            logit, reg = model(Xtr[b])
            loss = crit_cls(logit, ytr[b]) + REG_WEIGHT * crit_reg(reg, ptr[b])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        pr = window_probs(model, X_all[es_idx])
        if es_level == "subject":
            sids, ps = subject_scores(pr, ids_all[es_idx])
            score = accuracy_score([y_diag[s] for s in sids], (ps >= 0.5).astype(int))
        else:
            score = accuracy_score(y_all[es_idx], (pr >= 0.5).astype(int))
        if score > best:
            best, pat = score, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            pat += 1
            if pat >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model

## A · Out-of-fold subject embeddings for every partition

In [ ]:
parts = pd.read_csv(PARTITIONS, dtype={"subject_id": str})
parts = parts[parts.silo == "eeg"].copy()
parts["sid"] = parts.subject_id.map(norm_id)            # canonical id used to find windows
not_found = sorted(set(parts.sid) - set(unique_sids))
assert not not_found, f"partition subjects with no EEG windows: {not_found}"
print(f"partitions: {parts.partition.nunique()} | EEG subjects in partitions: {parts.sid.nunique()}")
N_PARTITIONS = parts.partition.nunique()
log = []

if RUN_SECTION_A:
    for r in range(N_PARTITIONS):
        if os.path.exists(f"{EMB_OUT}/eeg_emb_partition{r}.npz"):
            print(f"partition {r}: already done, skipping"); continue
        P = parts[parts.partition == r]
        tr_s = set(P[P.split == "train"].sid); va_s = set(P[P.split == "val"].sid)
        te_s = set(P[P.split == "test"].sid)
        assert not (tr_s & te_s) and not (va_s & te_s) and not (tr_s & va_s)
        tr_idx = np.where(np.isin(ids_all, list(tr_s)))[0]
        va_idx = np.where(np.isin(ids_all, list(va_s)))[0]
        sids = P.sid.to_numpy()                  # canonical ids (for finding windows)
        sids_out = P.subject_id.to_numpy()       # ids exactly as in partitions.csv (used by R1_2)
        embs = []
        t0 = time.time()
        for seed in ENC_SEEDS:
            model = train_encoder(tr_idx, va_idx, seed, es_level="subject")
            W = window_embeddings(model, X_all[np.isin(ids_all, sids)])
            wid = ids_all[np.isin(ids_all, sids)]
            embs.append(np.stack([W[wid == s].mean(axis=0) for s in sids]))
            te_idx = np.where(np.isin(ids_all, list(te_s)))[0]
            ts, tp = subject_scores(window_probs(model, X_all[te_idx]), ids_all[te_idx])
            log.append({"partition": r, "seed": seed,
                        "encoder_test_acc": accuracy_score([y_diag[s] for s in ts], (tp >= 0.5).astype(int))})
        np.savez(f"{EMB_OUT}/eeg_emb_partition{r}.npz", X=np.mean(embs, axis=0).astype(np.float32),
                 ids=sids_out.astype(str), encoder_train_ids=np.array(sorted(tr_s)),
                 encoder_es_ids=np.array(sorted(va_s)))
        print(f"partition {r}: train={len(tr_s)} val={len(va_s)} test={len(te_s)} "
              f"| saved {len(sids)} x 128 | {time.time()-t0:.0f}s")
    pd.DataFrame(log).to_csv(f"{EMB_OUT}/encoder_diagnostics.csv", index=False)
    print(pd.DataFrame(log).groupby("partition").encoder_test_acc.mean().describe())

## B · Controlled leakage comparison (same data, same model, only the split differs)

In [ ]:
if RUN_SECTION_B:
    sid_arr = np.array(unique_sids); ysub = np.array([y_diag[s] for s in sid_arr])
    recs = []
    for seed in LEAK_SEEDS:
        # ---- subject-wise ----
        skf = StratifiedKFold(n_splits=LEAK_FOLDS, shuffle=True, random_state=seed)
        oof_w = np.full(len(X_all), np.nan)
        for k, (trs, tes) in enumerate(skf.split(sid_arr, ysub)):
            tr_sub, es_sub = train_test_split(sid_arr[trs], test_size=INNER_VAL_FRAC,
                                              random_state=seed, stratify=ysub[trs])
            model = train_encoder(np.where(np.isin(ids_all, tr_sub))[0],
                                  np.where(np.isin(ids_all, es_sub))[0], seed, es_level="subject")
            te_idx = np.where(np.isin(ids_all, sid_arr[tes]))[0]
            oof_w[te_idx] = window_probs(model, X_all[te_idx])
        ss, sp = subject_scores(oof_w, ids_all)
        recs.append({"protocol": "subject-wise", "seed": seed,
                     "window_acc": accuracy_score(y_all, oof_w >= 0.5),
                     "window_f1": f1_score(y_all, oof_w >= 0.5, average="macro"),
                     "subject_acc": accuracy_score([y_diag[s] for s in ss], sp >= 0.5),
                     "subject_f1": f1_score([y_diag[s] for s in ss], sp >= 0.5, average="macro")})
        # ---- segment-wise (leaky) ----
        skf = StratifiedKFold(n_splits=LEAK_FOLDS, shuffle=True, random_state=seed)
        oof_w = np.full(len(X_all), np.nan)
        for k, (trw, tew) in enumerate(skf.split(X_all, y_all)):
            tr_w, es_w = train_test_split(trw, test_size=INNER_VAL_FRAC, random_state=seed, stratify=y_all[trw])
            model = train_encoder(tr_w, es_w, seed, es_level="window")
            oof_w[tew] = window_probs(model, X_all[tew])
        recs.append({"protocol": "segment-wise", "seed": seed,
                     "window_acc": accuracy_score(y_all, oof_w >= 0.5),
                     "window_f1": f1_score(y_all, oof_w >= 0.5, average="macro"),
                     "subject_acc": np.nan, "subject_f1": np.nan})
        print(recs[-2]); print(recs[-1])
    leak = pd.DataFrame(recs)
    leak.to_csv(f"{R1_OUT}/eeg_leakage_controlled.csv", index=False)
    summ = leak.groupby("protocol").agg(["mean", "std"])
    print(summ.round(4))

    from scipy import stats
    a = leak[leak.protocol == "segment-wise"].window_acc.to_numpy()
    b = leak[leak.protocol == "subject-wise"].window_acc.to_numpy()
    gap = a - b
    t, p = stats.ttest_rel(a, b)

    def fmt(x, pct=True): return f"{100*x:.1f}" if pct else f"{x:.3f}"
    m = summ
    macros = {
        "EEGleakSubjWinAcc":    fmt(m.loc["subject-wise", ("window_acc", "mean")]),
        "EEGleakSubjWinAccSD":  fmt(m.loc["subject-wise", ("window_acc", "std")]),
        "EEGleakSubjSubAcc":    fmt(m.loc["subject-wise", ("subject_acc", "mean")]),
        "EEGleakSubjSubAccSD":  fmt(m.loc["subject-wise", ("subject_acc", "std")]),
        "EEGleakSegWinAcc":     fmt(m.loc["segment-wise", ("window_acc", "mean")]),
        "EEGleakSegWinAccSD":   fmt(m.loc["segment-wise", ("window_acc", "std")]),
        "EEGleakGap":           fmt(gap.mean()),
        "EEGleakGapSD":         fmt(gap.std(ddof=1)),
        "EEGleakGapP":          f"{p:.3g}",
        "EEGleakNseeds":        str(len(LEAK_SEEDS)),
    }
    os.makedirs(f"{R1_OUT}/latex", exist_ok=True)
    with open(f"{R1_OUT}/latex/results_macros_eeg.tex", "w") as fh:
        fh.write("% auto-generated by R1_1_EEG_outoffold_embeddings.ipynb -- do not edit by hand\n")
        for k, v in macros.items():
            fh.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")
    print("saved ->", f"{R1_OUT}/latex/results_macros_eeg.tex")

    # figure: controlled comparison next to published MODMA numbers
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7.5, 4))
    bars = [("Ours, segment-wise\n(controlled)", a.mean(), a.std(ddof=1), "#d62728"),
            ("Ours, subject-wise\n(controlled)", b.mean(), b.std(ddof=1), "#1f77b4")]
    for i, (nm, mu, sd, c) in enumerate(bars):
        ax.bar(i, 100 * mu, yerr=100 * sd, color=c, capsize=5, edgecolor="k", width=0.6)
        ax.text(i, 100 * mu + 100 * sd + 1.5, f"{100*mu:.1f}%", ha="center", fontsize=9)
    ax.set_xticks(range(len(bars))); ax.set_xticklabels([x[0] for x in bars])
    ax.set_ylabel("Window-level accuracy (%)"); ax.set_ylim(0, 105)
    ax.set_title("MSCA-CNN-GRU on MODMA (3-channel): only the split protocol differs")
    fig.tight_layout()
    fig.savefig(f"{R1_OUT}/latex/fig_leakage_controlled.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("saved ->", f"{R1_OUT}/latex/fig_leakage_controlled.png")